In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_probe_epoch50_patience10 import ViViTBackbone, ViViTLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-11 21:22:40,381]::4094896549::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 50
print(1)
# Multi-seed setting
SEEDS = [42]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# ViViT Layer-wise Linear Probe | Multi-seed
# ============================================================
backbone_checkpoint_path = './save_swintransformer_folder/model_best_multiseed.pth'
probe_best_model_path = './save_swintransformer_folder/vivit_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
criterion = torch.nn.L1Loss().to(device)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True); return model

seed_results = []; global_best_mean_mae = float('inf'); global_best_seed = None; global_best_epoch = None

with torch.cuda.device(2):
    for seed in SEEDS:
        print('\n' + '=' * 80); print(f'Start ViViT probe | seed = {seed}'); print('=' * 80)
        set_seed(seed)
        train_generator = torch.Generator(); train_generator.manual_seed(seed)
        val_generator = torch.Generator(); val_generator.manual_seed(seed + 10000)
        train_dataloader = DataLoader(dataset=train_set_data,batch_size=batchsz,shuffle=True,num_workers=num_workerssz,worker_init_fn=seed_worker,generator=train_generator)
        val_dataloader = DataLoader(dataset=val_set_data,batch_size=batchsz,shuffle=False,num_workers=num_workerssz,worker_init_fn=seed_worker,generator=val_generator)

        backbone = load_backbone_checkpoint(ViViTBackbone(2,14,14,2,2,2,5,96,6,8,384,model=3), backbone_checkpoint_path)
        model = ViViTLinearProbe(backbone=backbone, num_outputs=5).to(device)
        optimizer = torch.optim.AdamW(model.probe_heads.parameters(), lr=lr)
        probe_names = list(model.probe_heads.keys())
        seed_best = {name:{'mae':float('inf'),'epoch':0} for name in probe_names}
        seed_best_mean_mae = float('inf'); no_improve_count = 0
        print('Backbone trainable params:', sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
        print('Probe dimensions:', model.probe_dims)

        for epoch in range(epochs):
            model.train(); train_abs_sum = {name:0.0 for name in probe_names}; train_count = {name:0 for name in probe_names}
            for audio, fullshot, big_five_data in tqdm(train_dataloader, desc=f'Probe Seed {seed} Train {epoch+1}/{epochs}'):
                fullshot = fullshot.permute(0,4,1,2,3); big_five_data = big_five_data.permute(0,2,1).squeeze()
                audio, fullshot, big_five_data = audio.to(device), fullshot.to(device), big_five_data.to(device)
                # backbone.eval()

                # with torch.no_grad():
                #     pred_original = backbone(fullshot, audio)

                #     features = backbone.forward_probe_features(fullshot, audio)

                #     pred_from_feature = backbone.mlp_head(features["final_fused"])

                # print((pred_original - pred_from_feature).abs().max())
                optimizer.zero_grad(); outputs = model(fullshot,audio)
                losses = {name:criterion(outputs[name],big_five_data) for name in probe_names}
                torch.stack([losses[name] for name in probe_names]).mean().backward(); optimizer.step()
                with torch.no_grad():
                    for name in probe_names:
                        train_abs_sum[name] += (outputs[name]-big_five_data).abs().sum().item(); train_count[name] += big_five_data.numel()
            train_mae = {name:train_abs_sum[name]/train_count[name] for name in probe_names}

            model.eval(); val_abs_sum = {name:0.0 for name in probe_names}; val_count = {name:0 for name in probe_names}
            with torch.no_grad():
                for audio, fullshot, big_five_data in tqdm(val_dataloader, desc=f'Probe Seed {seed} Val {epoch+1}/{epochs}'):
                    fullshot = fullshot.permute(0,4,1,2,3); big_five_data = big_five_data.permute(0,2,1).squeeze()
                    audio, fullshot, big_five_data = audio.to(device), fullshot.to(device), big_five_data.to(device)
                    outputs = model(fullshot,audio)
                    for name in probe_names:
                        val_abs_sum[name] += (outputs[name]-big_five_data).abs().sum().item(); val_count[name] += big_five_data.numel()
            val_mae = {name:val_abs_sum[name]/val_count[name] for name in probe_names}; mean_val_mae = float(np.mean([val_mae[name] for name in probe_names]))
            for name in probe_names:
                if val_mae[name] < seed_best[name]['mae']: seed_best[name]['mae'] = float(val_mae[name]); seed_best[name]['epoch'] = epoch + 1
            if mean_val_mae < seed_best_mean_mae:
                seed_best_mean_mae = mean_val_mae; no_improve_count = 0
                if mean_val_mae < global_best_mean_mae:
                    global_best_mean_mae = mean_val_mae; global_best_seed = seed; global_best_epoch = epoch + 1
                    torch.save({'seed':seed,'epoch':epoch+1,'probe_model':model.probe_heads.state_dict(),'mean_val_mae':mean_val_mae,'per_probe_val_mae':val_mae,'probe_dims':model.probe_dims},probe_best_model_path)
            else: no_improve_count += 1
            print(f'Probe Seed {seed} | Epoch {epoch+1:>3} | Mean MAE {mean_val_mae:.6f} | Mean 1-MAE {1.0-mean_val_mae:.6f}')
            for name in probe_names: print(f'  {name:<14} train={train_mae[name]:.6f} val={val_mae[name]:.6f} 1-MAE={1.0-val_mae[name]:.6f}')
            torch.cuda.empty_cache()
            if no_improve_count >= patience:
                print(f'Early stopping | seed={seed} | epoch={epoch+1} | patience={patience}'); break

        seed_results.append({'seed':seed,'results':seed_best})
        print('\nBest results for seed', seed)
        for name in probe_names: print(f'{name:<14} Best Epoch {seed_best[name]["epoch"]:>3} | MAE {seed_best[name]["mae"]:.6f} | 1-MAE {1.0-seed_best[name]["mae"]:.6f}')
        del model, backbone, optimizer, train_dataloader, val_dataloader, train_generator, val_generator; gc.collect(); torch.cuda.empty_cache()

print('\n' + '=' * 80); print('FINAL ViViT LINEAR PROBE RESULTS'); print('=' * 80)
probe_names = list(seed_results[0]['results'].keys())
for name in probe_names:
    maes = np.array([r['results'][name]['mae'] for r in seed_results],dtype=np.float64); scores = 1.0-maes
    print(f'{name:<14} MAE {maes.mean():.6f} ± {(maes.std(ddof=1) if len(maes)>1 else 0.0):.6f} | 1-MAE {scores.mean():.6f} ± {(scores.std(ddof=1) if len(scores)>1 else 0.0):.6f}')
print('-' * 80); print(f'Global best mean probe MAE: {global_best_mean_mae:.6f} | seed={global_best_seed} | epoch={global_best_epoch}'); print(f'Saved probe checkpoint: {probe_best_model_path}'); print('=' * 80)



Start ViViT probe | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable params: 0
Probe dimensions: {'block1': 96, 'block2': 96, 'block3': 96, 'block4': 96, 'block5': 96, 'block6': 96, 'final_video': 96, 'final_cross': 96, 'final_fused': 96}


Probe Seed 42 Train 1/50: 100%|█████████▉| 1496/1498 [02:14<00:00, 11.62it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Probe Seed 42 Val 1/50: 100%|██████████| 498/498 [00:46<00:00, 10.78it/s]


Probe Seed 42 | Epoch   1 | Mean MAE 0.092373 | Mean 1-MAE 0.907627
  block1         train=0.303019 val=0.086898 1-MAE=0.913102
  block2         train=0.163678 val=0.087239 1-MAE=0.912761
  block3         train=0.185546 val=0.091762 1-MAE=0.908238
  block4         train=0.296176 val=0.090278 1-MAE=0.909722
  block5         train=0.280434 val=0.088906 1-MAE=0.911094
  block6         train=0.149350 val=0.090285 1-MAE=0.909715
  final_video    train=0.259458 val=0.091672 1-MAE=0.908328
  final_cross    train=0.228720 val=0.114050 1-MAE=0.885950
  final_fused    train=0.261477 val=0.090270 1-MAE=0.909730


Probe Seed 42 Val 2/50: 100%|██████████| 498/498 [00:46<00:00, 10.81it/s]


Probe Seed 42 | Epoch   2 | Mean MAE 0.087248 | Mean 1-MAE 0.912752
  block1         train=0.035993 val=0.083614 1-MAE=0.916386
  block2         train=0.038028 val=0.083885 1-MAE=0.916115
  block3         train=0.043699 val=0.084529 1-MAE=0.915471
  block4         train=0.043744 val=0.084839 1-MAE=0.915161
  block5         train=0.041023 val=0.084178 1-MAE=0.915822
  block6         train=0.044322 val=0.084200 1-MAE=0.915800
  final_video    train=0.052185 val=0.085560 1-MAE=0.914440
  final_cross    train=0.112381 val=0.112219 1-MAE=0.887781
  final_fused    train=0.042236 val=0.082211 1-MAE=0.917789


...

Probe Seed 42 | Epoch  48 | Mean MAE 0.083655 | Mean 1-MAE 0.916345
  block1         train=0.021761 val=0.083508 1-MAE=0.916492
  block2         train=0.021828 val=0.083399 1-MAE=0.916601
  block3         train=0.021777 val=0.083421 1-MAE=0.916579
  block4         train=0.021928 val=0.083409 1-MAE=0.916591
  block5         train=0.021931 val=0.083492 1-MAE=0.916508
  block6         train=0.021854 val=0.083488 1-MAE=0.916512
  final_video    train=0.021988 val=0.083413 1-MAE=0.916587
  final_cross    train=0.053595 val=0.087741 1-MAE=0.912259
  final_fused    train=0.013721 val=0.081026 1-MAE=0.918974


Probe Seed 42 Val 49/50: 100%|██████████| 498/498 [00:44<00:00, 11.08it/s]


Probe Seed 42 | Epoch  49 | Mean MAE 0.083652 | Mean 1-MAE 0.916348
  block1         train=0.021778 val=0.083497 1-MAE=0.916503
  block2         train=0.021840 val=0.083444 1-MAE=0.916556
  block3         train=0.021786 val=0.083434 1-MAE=0.916566
  block4         train=0.021932 val=0.083424 1-MAE=0.916576
  block5         train=0.021936 val=0.083511 1-MAE=0.916489
  block6         train=0.021864 val=0.083496 1-MAE=0.916504
  final_video    train=0.022011 val=0.083464 1-MAE=0.916536
  final_cross    train=0.053055 val=0.087545 1-MAE=0.912455
  final_fused    train=0.013713 val=0.081052 1-MAE=0.918948


Probe Seed 42 Val 50/50: 100%|██████████| 498/498 [00:44<00:00, 11.09it/s]

Probe Seed 42 | Epoch  50 | Mean MAE 0.083662 | Mean 1-MAE 0.916338
  block1         train=0.021772 val=0.083589 1-MAE=0.916411
  block2         train=0.021825 val=0.083496 1-MAE=0.916504
  block3         train=0.021797 val=0.083469 1-MAE=0.916531
  block4         train=0.021913 val=0.083498 1-MAE=0.916502
  block5         train=0.021938 val=0.083535 1-MAE=0.916465
  block6         train=0.021865 val=0.083541 1-MAE=0.916459
  final_video    train=0.021997 val=0.083525 1-MAE=0.916475
  final_cross    train=0.052442 val=0.087268 1-MAE=0.912732
  final_fused    train=0.013727 val=0.081041 1-MAE=0.918959

Best results for seed 42
block1         Best Epoch  46 | MAE 0.083466 | 1-MAE 0.916534
block2         Best Epoch  34 | MAE 0.083382 | 1-MAE 0.916618
block3         Best Epoch  45 | MAE 0.083353 | 1-MAE 0.916647
block4         Best Epoch  34 | MAE 0.083365 | 1-MAE 0.916635
block5         Best Epoch  34 | MAE 0.083439 | 1-MAE 0.916561
block6         Best Epoch  45 | MAE 0.083400 | 1-MAE 0.9